# Improve Your Agent in 5 Minutes

The five-minute win: `pip install stateset-agents` -> grade your existing conversation logs -> get a curated training set, entirely offline, no GPU, no API key.

This mirrors [`examples/five_minute_demo.sh`](https://github.com/stateset/stateset-agents/blob/master/examples/five_minute_demo.sh): you bring conversation logs from an agent you built with *any* framework (OpenAI chat-completions format here), ingest them, grade them with a rule-based reward, and get back a curated JSONL of the high-scoring (prompt, response) pairs plus a graded report telling you what to do next.

**Runtime:** ~2 minutes. **No GPU required.**

## 1. Install

In [ ]:
%pip install --quiet stateset-agents
import stateset_agents
print("stateset-agents", stateset_agents.__version__)

## 2. Write sample conversation logs

In production these come from real users chatting with your agent (any framework — LangChain, raw OpenAI SDK, your own loop). For this demo we write three OpenAI chat-completions-format logs: one weak reply and two strong replies, so the grading step below produces a meaningful score spread.

In [ ]:
import json
from pathlib import Path

WORKDIR = Path("five_minute_demo")
WORKDIR.mkdir(exist_ok=True)

logs_path = WORKDIR / "logs.jsonl"
conversations = [
    {
        "messages": [
            {"role": "user", "content": "My order hasn't arrived yet, can you help?"},
            {"role": "assistant", "content": "No."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "I need a refund for my order, it arrived damaged."},
            {
                "role": "assistant",
                "content": (
                    "I'm sorry to hear your order arrived damaged — thank you for "
                    "letting us know. I'd be glad to help with your refund right "
                    "away. Please share your order number and I'll process it for "
                    "you today, no need to send the item back first."
                ),
            },
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Can you check on the status of my order? It's been a week."},
            {
                "role": "assistant",
                "content": (
                    "Of course, happy to help! I understand waiting a week is "
                    "frustrating. Please share your order number and I'll look up "
                    "the latest shipping status for you right now, and follow up "
                    "with tracking details."
                ),
            },
        ]
    },
]

with logs_path.open("w", encoding="utf-8") as f:
    for conv in conversations:
        f.write(json.dumps(conv) + "\n")

print(f"Wrote {len(conversations)} conversations -> {logs_path}")

## 3. Ingest into graded-history transcripts

`stateset-agents ingest` converts third-party log formats (OpenAI chat-completions or LangChain/LangGraph message dumps) into the `{"role", "content"}` JSONL transcripts the grade -> curate loop consumes.

In [ ]:
transcripts_dir = WORKDIR / "transcripts"

!python -m stateset_agents.cli ingest \
    --format openai \
    --input {logs_path} \
    --output {transcripts_dir}

## 4. Grade + curate with `stateset-agents improve`

One command runs the whole grade -> curate -> retrain loop: it grades every assistant turn with a rule-based reward (`customer_support` here — intent acknowledgement + brand voice + a safety multiplier, no API key needed), curates the pairs scoring at or above `--threshold` into `curated.jsonl`, and writes a `next_steps.md` telling you exactly how to train on it.

In [ ]:
improved_dir = WORKDIR / "improved"

!python -m stateset_agents.cli improve run \
    --transcripts {transcripts_dir} \
    --reward customer_support \
    --output {improved_dir}

## 5. Inspect the graded report and curated set

In [ ]:
summary = json.loads((improved_dir / "improve_summary.json").read_text())
print(f"transcripts graded : {summary['transcript_count']}")
print(f"assistant turns    : {summary['assistant_turn_count']}")
print(f"mean score         : {summary['mean_score']:.3f}")
print(f"threshold          : {summary['threshold']}")
print(f"curated examples   : {summary['curated_count']}")
print()
for t in summary["transcripts"]:
    print(f"  {t['name']:<20} mean={t['mean_score']:.3f}  above_threshold={t['above_threshold']}")

In [ ]:
curated_path = improved_dir / "curated.jsonl"
curated = [json.loads(line) for line in curated_path.read_text().splitlines() if line.strip()]
print(f"{len(curated)} curated example(s) in {curated_path}\n")
for ex in curated:
    print("prompt   :", ex["prompt"])
    print("response :", ex["response"])
    print("score    :", ex["score"])
    print()

### Score spread — weak vs. strong replies

In [ ]:
import matplotlib.pyplot as plt

names = [t["name"] for t in summary["transcripts"]]
means = [t["mean_score"] for t in summary["transcripts"]]
colors = ["#d62728" if m < summary["threshold"] else "#2ca02c" for m in means]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(names, means, color=colors)
ax.axhline(summary["threshold"], color="gray", linestyle="--", linewidth=1, label=f"threshold ({summary['threshold']})")
ax.set_ylabel("mean score")
ax.set_ylim(0, 1)
ax.set_title("customer_support reward: weak vs. strong replies")
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 6. Where to go next

Read `next_steps.md` for the exact commands (it's generated per-run, tailored to your reward and threshold):

In [ ]:
print((improved_dir / "next_steps.md").read_text())

## Keep going

* **Fine-tune on the curated set** (SFT, CPU-friendly to sanity-check): `python scripts/sft_from_curated.py --dataset ... --base-model Qwen/Qwen3.5-0.8B ...` — see `next_steps.md` above for the exact invocation.
* **Continue with RL (GSPO)** against the same reward, as a dry run (no GPU/API key needed to check the config):

  ```bash
  python examples/finetune_gspo.py --model qwen3.5-0.8b --task customer_support
  ```

  Add `--no-dry-run` once you're ready for a real, GPU-backed run.
* **Wire this into any MCP client** (Claude Code, Claude Desktop, etc.) so grade/curate/retrain becomes a tool call instead of a script:

  ```bash
  claude mcp add stateset-agents -- stateset-agents mcp
  ```

  See [`docs/MCP_SERVER.md`](https://github.com/stateset/stateset-agents/blob/master/docs/MCP_SERVER.md) for the full tool list.
* **More depth:** [`notebooks/grade_and_curate_demo.ipynb`](./grade_and_curate_demo.ipynb) (the same loop against a real fine-tuned checkpoint) and [`examples/getting_started/`](https://github.com/stateset/stateset-agents/tree/master/examples/getting_started) (the numbered GPU-free example ladder).